# 03 - Dataset, Patch and Training Analysis

This notebook summarizes the prepared CNN patch dataset and the first lightweight CNN training run.

In [ ]:
from pathlib import Path
import json

import cv2
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LABELS_PATH = PROJECT_ROOT / "data" / "labels" / "labels.csv"
PATCH_LABELS_PATH = PROJECT_ROOT / "data" / "processed" / "patch_labels.csv"
DATASET_SUMMARY_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_summary.json"
TRAINING_DIR = PROJECT_ROOT / "outputs" / "training"

labels = pd.read_csv(LABELS_PATH)
patch_labels = pd.read_csv(PATCH_LABELS_PATH)
patch_labels.head()

## Patch Dataset Summary

In [ ]:
with open(DATASET_SUMMARY_PATH, "r", encoding="utf-8") as f:
    dataset_summary = json.load(f)

dataset_summary

## Class Counts By Split

In [ ]:
split_counts = patch_labels.groupby(["split", "label"]).size().unstack(fill_value=0)
split_counts.plot(kind="bar", figsize=(8, 4), title="Patch Counts By Split")
plt.xlabel("Split")
plt.ylabel("Patch count")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
split_counts

## Class Counts By Original Scenario

In [ ]:
scenario_counts = patch_labels.groupby(["scenario_type", "label"]).size().unstack(fill_value=0)
scenario_counts.plot(kind="bar", stacked=True, figsize=(11, 5), title="Patch Classes By Scenario")
plt.xlabel("Scenario")
plt.ylabel("Patch count")
plt.xticks(rotation=35, ha="right")
plt.grid(axis="y", alpha=0.25)
scenario_counts

## Preview Correct and False Patches

In [ ]:
preview_paths = [
    PROJECT_ROOT / "outputs" / "patch-preview" / "correct_grid.png",
    PROJECT_ROOT / "outputs" / "patch-preview" / "false_grid.png",
]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, path in zip(axes, preview_paths):
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(path.name)
    ax.axis("off")
plt.tight_layout()

## Training History

In [ ]:
history_path = TRAINING_DIR / "history.csv"
if history_path.exists():
    history = pd.read_csv(history_path)
    display(history.tail())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["epoch"], history["train_loss"], label="train")
    axes[0].plot(history["epoch"], history["validation_loss"], label="validation")
    axes[0].set_title("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.25)
    axes[1].plot(history["epoch"], history["train_accuracy"], label="train")
    axes[1].plot(history["epoch"], history["validation_accuracy"], label="validation")
    axes[1].set_title("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.25)
    plt.tight_layout()
else:
    print("Training history not found yet. Run src/train_classifier.py first.")

## Test Metrics and Training Artifacts

In [ ]:
metrics_path = TRAINING_DIR / "test_metrics.json"
if metrics_path.exists():
    with open(metrics_path, "r", encoding="utf-8") as f:
        test_metrics = json.load(f)
    display(test_metrics)
else:
    print("Test metrics not found yet.")

for artifact in ["training_curves.png", "confusion_matrix.png", "sample_predictions.png"]:
    print(TRAINING_DIR / artifact)

## Notes

- Current data is synthetic only.
- False beacons are intentionally similar to true beacons, so top-1 mistakes are expected.
- The model should be re-tested on Unity frames before being used for closed-loop tracking.

## Phase 6 - Single-Frame Pipeline

Phase 6 connects preprocessing, bright-candidate detection, CNN patch classification and fused candidate ranking for one independent camera frame.

It writes a PID-ready JSON result and an annotated diagnostic image under `outputs/pipeline-test/`.


In [ ]:
import json
from pathlib import Path

pipeline_result_path = Path("../outputs/pipeline-test/result.json")
pipeline_image_path = Path("../outputs/pipeline-test/annotated_result.png")

if pipeline_result_path.exists():
    result = json.loads(pipeline_result_path.read_text(encoding="utf-8"))
    summary = {
        "target_found": result.get("target_found"),
        "status": result.get("status"),
        "candidate_count": result.get("candidate_count"),
        "selected_candidate_id": result.get("selected_candidate_id"),
        "cnn_probability": result.get("cnn_probability"),
        "cv_baseline_score": result.get("cv_baseline_score"),
        "fused_score": result.get("fused_score"),
        "control_error_x": result.get("control_error_x"),
        "control_error_y": result.get("control_error_y"),
    }
    summary
else:
    print("Run src/pipeline.py first to create outputs/pipeline-test/result.json")


In [ ]:
from IPython.display import Image, display

if pipeline_image_path.exists():
    display(Image(filename=str(pipeline_image_path)))
else:
    print("Run src/pipeline.py first to create outputs/pipeline-test/annotated_result.png")


## Phase 7 - Temporal Verification And Tracking

Phase 7 adds ordered-frame memory on top of the Phase 6 single-frame pipeline. The tracker uses candidate association, persistence-based temporal verification, and a constant-velocity Kalman filter to produce stable coordinates and lock-state outputs.

Generated smoke-test artifacts are expected under `outputs/tracking-test/`:

- `tracking_results.csv`
- `tracking_summary.json`
- `annotated_tracking.mp4`


In [ ]:
from pathlib import Path
import json
import pandas as pd

tracking_dir = Path("../outputs/tracking-test")
summary_path = tracking_dir / "tracking_summary.json"
results_path = tracking_dir / "tracking_results.csv"

print("summary exists:", summary_path.exists())
print("results exists:", results_path.exists())
print("video exists:", (tracking_dir / "annotated_tracking.mp4").exists())


In [ ]:
if summary_path.exists():
    tracking_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    tracking_summary
else:
    print("Run src/tracker.py first to generate the Phase 7 summary.")


In [ ]:
if results_path.exists():
    tracking_results = pd.read_csv(results_path)
    display(tracking_results[[
        "frame_index",
        "lock_state",
        "measurement_available",
        "using_prediction_only",
        "missed_frames",
        "target_found",
        "filtered_x_px",
        "filtered_y_px",
        "predicted_x_px",
        "predicted_y_px",
    ]].head(18))
else:
    print("Run src/tracker.py first to generate tracking_results.csv.")


### Phase 7 Notes

The smoke sequence intentionally hides the target for two frames. Healthy behavior is `LOCKED -> COASTING -> LOCKED`: the tracker should use prediction-only output during the short dropout and return to measured tracking once the beacon appears again.

Current smoke-test result:

```text
Total frames: 18
Measurements: 16
Locked frames: 14
Coasting frames: 2
Lost frames: 0
Maximum consecutive missed frames: 2
Mean association distance: 1.109044 px
```

No real tracking accuracy is claimed from this smoke test because it does not yet use ground-truth trajectory evaluation.


## Phase 8 - Unity Baseline Evaluation

Phase 8 evaluates the first continuous Unity sequence without retraining the CNN. The goal is to measure the baseline domain gap between Python-generated training data and Unity-rendered frames.

Sequence used:

`data/raw/unity/smooth_horizontal01/sequence_001/`

Evaluation outputs:

`outputs/unity-evaluation/smooth_horizontal_01/`


In [ ]:
from pathlib import Path
import json
import pandas as pd

unity_eval_dir = Path("../outputs/unity-evaluation/smooth_horizontal_01")
summary_file = unity_eval_dir / "evaluation_summary.json"
validation_file = unity_eval_dir / "dataset_validation.json"
detection_file = unity_eval_dir / "detection_metrics.json"
tracking_file = unity_eval_dir / "tracking_metrics.json"
performance_file = unity_eval_dir / "performance_metrics.json"
domain_gap_file = unity_eval_dir / "domain_gap_report.json"
predictions_file = unity_eval_dir / "frame_predictions.csv"

for file in [summary_file, validation_file, detection_file, tracking_file, performance_file, domain_gap_file, predictions_file]:
    print(file.name, file.exists())


In [ ]:
if summary_file.exists():
    unity_summary = json.loads(summary_file.read_text(encoding="utf-8"))
    unity_summary
else:
    print("Run src/evaluate_unity_sequence.py first.")


In [ ]:
if detection_file.exists() and tracking_file.exists() and domain_gap_file.exists():
    detection = json.loads(detection_file.read_text(encoding="utf-8"))
    tracking = json.loads(tracking_file.read_text(encoding="utf-8"))
    domain_gap = json.loads(domain_gap_file.read_text(encoding="utf-8"))
    pd.DataFrame([
        {"metric": "candidate_recall", "value": detection.get("candidate_recall")},
        {"metric": "accepted_detection_recall", "value": detection.get("accepted_detection_recall")},
        {"metric": "filtered_mae_px", "value": tracking.get("coordinate_metrics", {}).get("filtered_mae_px")},
        {"metric": "locked_frame_percentage", "value": tracking.get("locked_frame_percentage")},
        {"metric": "average_candidates_per_frame", "value": domain_gap.get("average_number_of_candidates")},
        {"metric": "most_common_failure_reason", "value": domain_gap.get("most_common_failure_reason")},
    ])
else:
    print("Unity metric files are not available yet.")


In [ ]:
if predictions_file.exists():
    predictions = pd.read_csv(predictions_file)
    display(predictions[[
        "frame_index",
        "target_x",
        "target_y",
        "phase6_target_found",
        "phase6_x_px",
        "phase6_y_px",
        "candidate_count",
        "confidence",
        "lock_state",
        "failure_reason",
    ]].head(10))
else:
    print("frame_predictions.csv is not available yet.")


### Phase 8 Interpretation

The Unity sequence itself is valid: 300 continuous PNG frames at 640 x 480 with readable labels. The baseline model, however, shows a strong domain gap. It often selects bright Unity artifacts instead of the labelled beacon.

Current baseline:

```text
Candidate recall: 0.096667
Accepted-detection recall: 0.0
Filtered MAE: 288.899709 px
Average candidates per frame: 8.086667
Most common failure: selected_candidate_far_from_ground_truth
```

This sequence should stay as baseline evaluation data. Do not use it alone for CNN fine-tuning because splitting one continuous sequence into train/test frames would leak temporal information.


### Phase 8 Readiness Upgrade

The Unity evaluator is now ready for higher-resolution and multi-sequence datasets without retraining the CNN. Resolution can be auto-detected from the frames, labels are checked for duplicate/missing/out-of-bounds rows, and batch mode creates combined metrics.

Command used on the current Unity data:

~~~powershell
.\.venv\Scripts\python.exe src\evaluate_unity_sequence.py --batch --sequence-dir data\raw\unity --config configs\unity.yaml --fps 30 --coordinate-origin top-left --output outputs\unity-evaluation --output-scale 2 --device cpu
~~~

Outputs added:

~~~text
outputs/unity-evaluation/final_metrics.csv
outputs/unity-evaluation/final_summary.json
outputs/unity-evaluation/smooth_horizontal_01/annotated_tracking.mp4
~~~

Current run summary:

~~~text
Sequences discovered: 1
Successful sequences: 1
Failed sequences: 0
Source resolution: 640 x 480
Annotated MP4 resolution: 1280 x 960
Candidate recall: 0.096667
Accepted-detection recall: 0.0
Filtered MAE: 288.899709 px
Most common failure: selected_candidate_far_from_ground_truth
~~~

The larger MP4 is for presentation readability only. Real image quality still depends on Unity exporting higher-resolution frames, such as 1280 x 720, with labels in the same pixel coordinate space.


### Unity Base 2400 Training Result

A new Unity dataset was inspected under data/raw/unity/cont_dataset_2400/unity_base_2400/. It contains 8 labelled sequences, 300 frames each, for 2400 total frames at 1280 x 720.

Scenarios:

~~~text
smooth_horizontal
smooth_vertical
diagonal
curved
speed_variation
disturbance
beacon_dropout
reacquisition
~~~

A label timing check found that label frame t best matches image frame t+1, so Unity patch preparation and post-training evaluation used --label-frame-offset 1.

Patch dataset prepared with src/prepare_unity_patches.py:

~~~text
Total patches: 5624
Correct patches: 1406
False patches: 4218
Train: 1020 correct, 3060 false
Validation: 184 correct, 552 false
Test: 202 correct, 606 false
~~~

Unity-domain classifier training produced:

~~~text
models/checkpoints/unity_base_2400/best_classifier.pt
outputs/training/unity_base_2400/test_metrics.json
outputs/training/unity_base_2400/training_curves.png
~~~

Patch-level held-out test result:

~~~text
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1-score: 1.0000
ROC-AUC: 1.0000
~~~

Full-frame Phase 8 comparison:

~~~text
Before Unity training:
  Candidate recall: 0.033074
  Accepted recall: 0.000000
  Filtered MAE: 475.900149 px

After Unity training with corrected label offset:
  Candidate recall: 0.392973
  Accepted recall: 0.144187
  Filtered MAE: 59.11329 px
~~~

Conclusion: Unity-domain training helped, but the remaining bottleneck is candidate detection. If OpenCV does not produce a candidate near the real beacon, the CNN cannot recover it.


## Official Unity 1600x900 V2 Results\n\nValidated 12 Unity sequences with 3600 total frames at 1600x900. Scenarios include smooth, vertical, diagonal, curved, speed variation, disturbance, dropout, reacquisition, dim beacon, multiple beacon, target absent, and false beacon/star-heavy.\n\nBaseline evaluation: candidate recall 0.912883, accepted recall 0.377689, filtered MAE 303.05556 px.\n\nPatch preparation: 10772 total patches, 2693 correct, 8079 false. Split logic was updated so target-absent-only data does not become the validation set.\n\nCNN training: early stopped after 12 epochs. Test accuracy 0.8639, precision 0.7034, recall 0.7876, F1 0.7432, ROC-AUC 0.9299.\n\nTrained full-frame evaluation: candidate recall 0.999650, accepted recall 0.074614, filtered MAE 5.919378 px, locked frames 44.583333%, effective CPU FPS 8.093595.\n\nInterpretation: candidate detection is now strong and localization error is low. Remaining work is accepted detection confidence/temporal/tracker tuning.\n

## Official Unity 1600x900 9k Results\n\nDataset inspected: data/raw/unity/official_1600x900/official_1600x900\n\n- 30 sequences, 9000 frames total\n- 300 frames per sequence\n- 1600x900 resolution\n- all frame sequences continuous\n- mostly smooth ground truth; sequence_020 has one large step above 120 px\n\nPatch prep output: data/processed/official_1600x900_patches\n\n- total patches: 9271\n- correct patches: 8132\n- false patches: 1139\n\nTraining output: outputs/training/official_1600x900_final\nCheckpoint: models/checkpoints/official_1600x900_final/best_classifier.pt\n\nPatch-level test metrics: accuracy 1.0000, precision 1.0000, recall 1.0000, F1 1.0000, ROC-AUC 1.0000.\n\nFull-frame evaluation output: outputs/unity-evaluation/official_1600x900_final_trained\n\n- candidate recall: 0.994643\n- accepted detection recall: 0.078690\n- filtered MAE: 2.277340 px\n- locked frames: 39.022222%\n- effective FPS: 8.189958\n\nMain finding: candidate localization is strong, but full-frame acceptance is still too strict/low-confidence. Next tuning should focus on accepted recall and rejected-candidate analysis.\n